In [4]:
import glob
import json
import os
import pandas as pd

path_repo = "data/repos"
path_dependent = "data/dependents"

repo_commit_info = {
    "alpaca_eval": "f19c323d8309d0d6f306bd26597db44fc6c62d57",
    "ann-benchmarks": "36681911472059cd4cb9fcac1686709201a2cbc0",
    "ARES": "c7c9018a755faf8347c4da415632bae1593ef104",
    "AutoRAG": "8c72159df28006e0d6c548bbf0c133dde2650fd8",
    "beir": "6ef8c9097ebfb203ad360bd64e0cfb93e64f4a44",
    "CipherChat": "540b95e6afb54e7776f08ae2c094f9d7ff7e7b1f",
    "bigcode-evaluation-harness": "6116c6a9a5672c69bd624373cfbc8938b7acc249",
    "COMET": "49c348dd0ff569c4d7c3f2b7c720fd5696f8de59",
    "deepeval": "f296dc03b76b5fb5cc124d2dbb10367af0a5777d",
    "DomainBed": "b93c22a1cfc3b2428398272c1a116c8de1f4139e",
    "EvalAI": "8d1444f233a84f57b4a6b3db2bc7503da4cc7194",
    "evalplus": "d362e933265c3e7e3df8101c930a89c3c470cd9f",
    "evals": "cdb8ce9547e68b8e5e4520b6a162294c06865c0f",
    "evalscope": "2a49bccc14f69b9a3464f16d4672823a0a7bcb98",
    "evaluate": "5aa3982a9a8c86e506860e381d428a64b0cce73b",
    "evidently": "609cc2afbfa28c6b37be725b9f1f61bd18136990",
    "GAOKAO-Bench": "6dbb24f8d8439041e5431c4c184a582182a6ce9c",
    "giskard": "6808a562cc1c565e6cf88aeb1fd1f77d28fe76ba",
    "helm": "68b6aa9a01598aa0d371507c4c7299260eb9c4e6",
    "human-eval": "6d43fb980f9fee3c892a914eda09951f772ad10d",
    "inspect_ai": "70fb9f6885db89779cd637619bbba614e7b0271a",
    "intellagent": "2680da4545b87717a8dbc869738238022ca84ee4",
    "jiwer": "c1b0d5e005431f5ce4fa6797f48639a8ccaa5042",
    "lm-evaluation-harness": "9f152e0b89e777a17a21e9207da2ae4c1ea8beec",
    "lighteval": "9771f3a02ba28a57f8777902d0c36ea7dd445026",
    "llmperf": "f1d6bed47e4501b0e371082b41601b59ab55269f",
    "lmms-eval": "c4a31289c56a673aba99d2015a1db1decf5308e7",
    "meltingpot": "f280a35a4ffe5a7613b5c4a062a66fd18c64876c",
    "Metaworld": "cdafedfaaa896472fb7e76e2fe103e120b67a9b6",
    "mir_eval": "8db0b3812e2032544c1fc00d02d4256cab043f3d",
    "inference": "cc7d6d8eabc5cc60f8b2e540f25f1edd2a9f6cd3",
    "ogb": "61e9784ca76edeaa6e259ba0f836099608ff0586",
    "ollama-grid-search": "eaf4edb3ba536764c45ec7863daf1aed6dca415a",
    "OpenCompass": "8379c4bca0e19aab4059a0eb7a739fa58f3b2759",
    "opik": "72628f77bade9dfb508cc22ce71362bb0e6ed132",
    "overcooked_ai": "739950a079cdaed5a44fcc662efc40244c205d06",
    "prometheus-eval": "dcfb44272d5d0414832f5dbb8c2a05ebc2614234",
    "promptbench": "fcda538bd779ad11612818e0645a387a462b5c3b",
    "pykeen": "42bfa99a17ee489d0d54dc6332d5cc71fc0ded08",
    "Quantus": "68a19a1125f98f0f596c12d102fbc15d31515c80",
    "ragas": "5d59549ad5ef511f621502c563bc55ac5aeb9188",
    "RAGChecker": "6091f08c00e676e87a970f2aeb4a23a484746348",
    "ranx": "f60b1ac230ab97bd3c6ad2c210f79882293b306d",
    "reward-bench": "09eb3b5dd9d2157adc854c693a6a38105a8b3c87",
    "RLBench": "02720bba4c73fe02eb75df946b8791b806028a9d",
    "SimplerEnv": "4ab7178e83e84ee06894034ec6dbf9e7aad1e882",
    "simple-evals": "3ec4e9b5ae3931a1858580e2fd3ce80c7fcbe1d9",
    "speech-to-text-benchmark": "42fdc1b2f8cb912551b52e4b637feb789bb68897",
    "model-analysis": "b703eae15ebb46ca40923e2daa955750e3893a20",
    "benchmark": "68cf9dffe5df4eedac42512dc92598fffff335ae",
    "trulens": "57df4120d19dd1a0b59efc6cccb0de205ac403e0",
    "TrustLLM": "16ce657080c24b32729ad824ac73bcf3e4e5f301",
    "VBench": "d2c30cab5497472af4798bc86f6eb47659381cb4",
    "VLMEvalKit": "50c954aad1d92dab9bf5d23f62576c994b4a675f",
    "evalai-cli": "6426f26a9add307841665215e795d4dca86ee362",
}

usage_pattern_mapping = {
    "alpaca_eval": "^\s*import\s+alpaca_eval|^\s*from\s+[\.]*alpaca_eval(\.[\w]*)*\s+import|alpaca_eval (evaluate|analyze_evaluators|make_leaderboard)", # https://github.com/tatsu-lab/alpaca_eval/blob/f19c323d8309d0d6f306bd26597db44fc6c62d57/setup.py#L85
    "ann-benchmarks": "^\s*import\s+ann_benchmarks|^\s*from\s+[\.]*ann_benchmarks(\.[\w]*)*\s+import",
    "ARES": "^\s*import\s+ares|^\s*from\s+[\.]*ares(\.[\w]*)*\s+import",
    "AutoRAG": "^\s*import\s+autorag|^\s*from\s+[\.]*autorag(\.[\w]*)*\s+import|autorag (evaluate|run_api|run_web|dashboard|extract_best_config|restart_evaluate|validate)|docker pull autoraghq/autorag", # https://github.com/Marker-Inc-Korea/AutoRAG/blob/23c81972136868bec463c1c607407c2890f5cc9e/autorag/pyproject.toml#L146
    "beir": "^\s*import\s+beir|^\s*from\s+[\.]*beir(\.[\w]*)*\s+import",
    "bigcode-evaluation-harness": "^\s*import\s+bigcode_eval|^\s*from\s+[\.]*bigcode_eval(\.[\w]*)*\s+import|docker pull ghcr\.io/bigcode-project/evaluation-harness",
    "COMET": "^\s*import\s+comet|^\s*from\s+[\.]*comet(\.[\w]*)*\s+import|comet-(train|score|compare|mbr)", # https://github.com/Unbabel/COMET/blob/49c348dd0ff569c4d7c3f2b7c720fd5696f8de59/pyproject.toml#L32
    "deepeval": "^\s*import\s+deepeval|^\s*from\s+[\.]*deepeval(\.[\w]*)*\s+import|deepeval (enable_grpc_logging|login|logout|recommend|set_azure_openai_embedding_env|set_azure_openai_env|set-confident-region|set_deepseek_model_env|set_gemini_model_env|set_grok_model_env|set_litellm_model_env|set_local_embeddings_env|set_local_model_env|set_moonshot_model_env|set_ollama_embeddings_env|set_ollama_model_env|set_openai_env|test|unset_azure_openai_embedding_env|unset_azure_openai_env|unset_deepseek_model_env|unset_gemini_model_env|unset_grok_model_env|unset_litellm_model_env|unset_local_embeddings_env|unset_local_model_env|unset_moonshot_model_env|unset_ollama_embeddings_env|unset_ollama_model_env|unset_openai_env|view)", # https://github.com/confident-ai/deepeval/blob/67f96688113645fbc9cce9aca2487d8e9dbf7711/pyproject.toml#L13
    "DomainBed": "^\s*import\s+domainbed|^\s*from\s+[\.]*domainbed(\.[\w]*)*\s+import",
    "EvalAI": "^\s*import\s+evalai|^\s*from\s+[\.]*evalai(\.[\w]*)*\s+import|evalai (challenge|download_file|get_token|host|login|phase|push|set_token|submission|teams)", # https://github.com/Cloud-CV/evalai-cli/blob/6426f26a9add307841665215e795d4dca86ee362/setup.py#L69
    "evalplus": "^\s*import\s+evalplus|^\s*from\s+[\.]*evalplus(\.[\w]*)*\s+import|evalplus\.(codegen|evalperf|evaluate|inputgen|sanitize|syncheck)", # https://github.com/evalplus/evalplus/blob/13845c6f446f35cebbb123c5add72e841491c2c7/setup.cfg#L40
    "evals": "^\s*import\s+evals|^\s*from\s+[\.]*evals(\.[\w]*)*\s+import|oaieval", # https://github.com/openai/evals/blob/cdb8ce9547e68b8e5e4520b6a162294c06865c0f/pyproject.toml#L61
    "evalscope": "^\s*import\s+(evalscope|llmuses)|^\s*from\s+[\.]*(evalscope|llmuses)(\.[\w]*)*\s+import|evalscope (app|eval|perf|server)", # https://github.com/modelscope/evalscope/blob/5f32bfabdcb3d722e8fbee0ed15eeb2b07a358fc/setup.py#L192
    "evaluate": "^\s*import\s+evaluate|^\s*from\s+[\.]*evaluate(\.[\w]*)*\s+import|evaluate-cli create", # https://github.com/huggingface/evaluate/blob/f99652721de8d8c7f4526cb00156f6c3cb3c49e3/setup.py#L140
    "evidently": "^\s*import\s+evidently|^\s*from\s+[\.]*evidently(\.[\w]*)*\s+import|evidently (demo_project|report|ui)", # https://github.com/evidentlyai/evidently/blob/0da3a24284e72a5dd117e1ce261586005d11a6b3/setup.py#L123
    "GAOKAO-Bench": "^\s*import\s+bench_function|^\s*from\s+[\.]*bench_function(\.[\w]*)*\s+import", # https://github.com/OpenLMLab/GAOKAO-Bench/blob/c214b47c7046d4ef048c77f0a7c5b545dc8d337b/Bench/bench_function.py
    "giskard": "^\s*import\s+giskard|^\s*from\s+[\.]*giskard(\.[\w]*)*\s+import",
    "helm": "^\s*import\s+helm|^\s*from\s+[\.]*helm(\.[\w]*)*\s+import|helm-(create-plots|run|server|summarize)|crfm-proxy-(cli|server)", # https://github.com/stanford-crfm/helm/blob/main/setup.cfg#L390
    "human-eval": "^\s*import\s+human_eval|^\s*from\s+[\.]*human_eval(\.[\w]*)*\s+import|evaluate_functional_correctness",
    "inspect_ai": "^\s*import\s+inspect_ai|^\s*from\s+[\.]*inspect_ai(\.[\w]*)*\s+import|inspect (cache|eval|info|log|sandbox|score|trace|view)", # https://github.com/UKGovernmentBEIS/inspect_ai/blob/main/pyproject.toml#L136
    "intellagent": "^\s*import\s+simulator|^\s*from\s+[\.]*simulator(\.[\w]*)*\s+import",
    "jiwer": "^\s*import\s+jiwer|^\s*from\s+[\.]*jiwer(\.[\w]*)*\s+import",
    "lm-evaluation-harness": "^\s*import\s+lm_eval|^\s*from\s+[\.]*lm_eval(\.[\w]*)*\s+import|(lm_eval|lm-eval)", # https://github.com/EleutherAI/lm-evaluation-harness/blob/314f717677d4aa0c706fc1c6bbd3c801d321384a/pyproject.toml#L51
    "lighteval": "^\s*import\s+lighteval|^\s*from\s+[\.]*lighteval(\.[\w]*)*\s+import|lighteval (accelerate|baseline|create|custom|endpoint|inference_endpoint|inference_providers|inspect|list|litellm|nanotron|sglang|tasks|tgi|vllm)", # https://github.com/huggingface/lighteval/blob/9771f3a02ba28a57f8777902d0c36ea7dd445026/pyproject.toml#L124
    "llmperf": "^\s*import\s+llmperf|^\s*from\s+[\.]*llmperf(\.[\w]*)*\s+import",
    "lmms-eval": "^\s*import\s+lmms_eval|^\s*from\s+[\.]*lmms_eval(\.[\w]*)*\s+import",
    "meltingpot": "^\s*import\s+meltingpot|^\s*from\s+[\.]*meltingpot(\.[\w]*)*\s+import",
    "Metaworld": "^\s*import\s+metaworld|^\s*from\s+[\.]*metaworld(\.[\w]*)*\s+import",
    "mir_eval": "^\s*import\s+mir_eval|^\s*from\s+[\.]*mir_eval(\.[\w]*)*\s+import",
    "ogb": "^\s*import\s+ogb|^\s*from\s+[\.]*ogb(\.[\w]*)*\s+import",
    "OpenCompass": "opencompass",
    "opik": "^\s*import\s+opik|^\s*from\s+[\.]*opik(\.[\w]*)*\s+import|opik (configure|healthcheck|proxy)|docker pull ghcr.io/comet-ml/opik/opik-(backend|guardrails-backend|frontend|python-backend|sandbox-executor-python)", # https://github.com/comet-ml/opik/blob/2a2031c3c9bfc9da1cfb9e2b8b49fead46445891/sdks/python/setup.py#L67
    "overcooked_ai": "^\s*import\s+human_aware_rl|^\s*from\s+[\.]*human_aware_rl(\.[\w]*)*\s+import|^\s*import\s+overcooked_ai_py|^\s*from\s+[\.]*overcooked_ai_py(\.[\w]*)*\s+import|^\s*import\s+overcooked_demo|^\s*from\s+[\.]*overcooked_demo(\.[\w]*)*\s+import",
    "prometheus-eval": "^\s*import\s+prometheus_eval|^\s*from\s+[\.]*prometheus_eval(\.[\w]*)*\s+import",
    "promptbench": "^\s*import\s+promptbench|^\s*from\s+[\.]*promptbench(\.[\w]*)*\s+import",
    "pykeen": "^\s*import\s+pykeen|^\s*from\s+[\.]*pykeen(\.[\w]*)*\s+import|pykeen (experiments|optimize|tokenize)", # https://github.com/pykeen/pykeen/blob/dbdfb0261002d1f4405710aba190f298424c72e6/pyproject.toml#L164
    "Quantus": "^\s*import\s+quantus|^\s*from\s+[\.]*quantus(\.[\w]*)*\s+import",
    "ragas": "^\s*import\s+ragas|^\s*from\s+[\.]*ragas(\.[\w]*)*\s+import",
    "RAGChecker": "^\s*import\s+ragchecker|^\s*from\s+[\.]*ragchecker(\.[\w]*)*\s+import|ragchecker-cli", # https://github.com/amazon-science/RAGChecker/blob/6091f08c00e676e87a970f2aeb4a23a484746348/pyproject.toml#L24
    "ranx": "^\s*import\s+ranx|^\s*from\s+[\.]*ranx(\.[\w]*)*\s+import",
    "reward-bench": "rewardbench",
    "RLBench": "^\s*import\s+rlbench|^\s*from\s+[\.]*rlbench(\.[\w]*)*\s+import|rlbench-generate-dataset", # https://github.com/stepjam/RLBench/blob/02720bba4c73fe02eb75df946b8791b806028a9d/setup.py#L51
    "SimplerEnv": "^\s*import\s+simpler_env|^\s*from\s+[\.]*simpler_env(\.[\w]*)*\s+import",
    "model-analysis": "^\s*import\s+tensorflow_model_analysis|^\s*from\s+[\.]*tensorflow_model_analysis(\.[\w]*)*\s+import",
    "benchmark": "^\s*import\s+torchbenchmark|^\s*from\s+[\.]*torchbenchmark(\.[\w]*)*\s+import|^\s*import\s+userbenchmark|^\s*from\s+[\.]*userbenchmark(\.[\w]*)*\s+import",
    "trulens": "^\s*import\s+trulens|^\s*from\s+[\.]*trulens(\.[\w]*)*\s+import",
    "TrustLLM": "^\s*import\s+trustllm|^\s*from\s+[\.]*trustllm(\.[\w]*)*\s+import",
    "VBench": "^\s*import\s+vbench|^\s*from\s+[\.]*vbench(\.[\w]*)*\s+import|vbench (evaluate|static_filter)", # https://github.com/Vchitect/VBench/blob/89d362979d0d9247882fd8622fc9c0e4fb4cf440/setup.py#L48
    "VLMEvalKit": "^\s*import\s+vlmeval|^\s*from\s+[\.]*vlmeval(\.[\w]*)*\s+import|vlmutil (check|circular|dlist|eval|localize|merge_pkl|missing|mlist|run|scan)", # https://github.com/open-compass/VLMEvalKit/blob/44d698f846b8f74e34a999e9cc3af52f400403d3/setup.py#L108
}

<>:68: SyntaxWarning: invalid escape sequence '\s'
<>:69: SyntaxWarning: invalid escape sequence '\s'
<>:70: SyntaxWarning: invalid escape sequence '\s'
<>:71: SyntaxWarning: invalid escape sequence '\s'
<>:72: SyntaxWarning: invalid escape sequence '\s'
<>:73: SyntaxWarning: invalid escape sequence '\s'
<>:74: SyntaxWarning: invalid escape sequence '\s'
<>:75: SyntaxWarning: invalid escape sequence '\s'
<>:76: SyntaxWarning: invalid escape sequence '\s'
<>:77: SyntaxWarning: invalid escape sequence '\s'
<>:78: SyntaxWarning: invalid escape sequence '\s'
<>:79: SyntaxWarning: invalid escape sequence '\s'
<>:80: SyntaxWarning: invalid escape sequence '\s'
<>:81: SyntaxWarning: invalid escape sequence '\s'
<>:82: SyntaxWarning: invalid escape sequence '\s'
<>:83: SyntaxWarning: invalid escape sequence '\s'
<>:84: SyntaxWarning: invalid escape sequence '\s'
<>:85: SyntaxWarning: invalid escape sequence '\s'
<>:86: SyntaxWarning: invalid escape sequence '\s'
<>:87: SyntaxWarning: invalid e

In [ ]:
import os
import subprocess
import shutil
import time
import platform
import pandas as pd

df_harness = pd.read_csv("data/evaluation_harness_metadata.csv", encoding="utf-8")
df_harness['repo'] = df_harness['repo'].apply(lambda x: x.split(","))
df_harness = df_harness.explode('repo').reset_index(drop=True)

# Define path for repositories
path_repo = "data/repos"

# Make sure the base directory exists
os.makedirs(path_repo, exist_ok=True)


def get_latest_commit_hash(repo_path):
    """
    Get the latest commit hash from a git repository.
    """
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_path,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except Exception as e:
        print(f"Error getting commit hash for {repo_path}: {str(e)}")
        return None


def get_commit_info(repo_path):
    """
    Get commit hash, date, and message for the latest commit.

    Args:
        repo_path (str): Path to the git repository

    Returns:
        dict: Dictionary containing 'hash', 'date', and 'message' keys,
              or None if an error occurs
    """
    try:
        # Get commit hash
        hash_result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_path,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True,
        )
        commit_hash = hash_result.stdout.strip()
        return commit_hash

    except subprocess.CalledProcessError as e:
        print(f"Git command failed for {repo_path}: {e.stderr.strip()}")
        return None
    except FileNotFoundError:
        print(f"Git not found or repository path invalid: {repo_path}")
        return None
    except Exception as e:
        print(f"Error getting commit info for {repo_path}: {str(e)}")
        return None


def safe_remove_directory(path):
    """
    Safely remove a directory, handling Windows permission errors
    by attempting multiple times with delays.
    """
    if not os.path.exists(path):
        return

    # First, try to make files writable in case they're read-only
    try:
        if platform.system() == "Windows":
            subprocess.run(
                ["attrib", "-R", path + "\\*.*", "/S"], shell=True, check=False
            )
    except:
        pass  # Ignore errors from this step

    max_attempts = 3
    for attempt in range(max_attempts):
        try:
            print(f"Cleanup attempt {attempt+1} for {path}")
            shutil.rmtree(path, ignore_errors=True)
            if not os.path.exists(path):
                print(f"Successfully removed {path}")
                return

            print(
                f"Directory still exists after removal attempt {attempt+1}. Waiting..."
            )
            time.sleep(3)
        except Exception as e:
            print(f"Error during cleanup attempt {attempt+1}: {str(e)}")
            if attempt < max_attempts - 1:
                time.sleep(3)

    print(
        f"Warning: Failed to remove {path} after {max_attempts} attempts. Continuing..."
    )


def clone_problematic_quantus(repo_path, url):
    """
    Special handling for the Quantus repository which has problematic files
    using a sparse checkout approach.
    """
    # Initialize git repo
    os.makedirs(repo_path, exist_ok=True)

    try:
        # Initialize empty git repo
        subprocess.run(["git", "init"], cwd=repo_path, check=True)

        # Add remote
        subprocess.run(
            ["git", "remote", "add", "origin", url], cwd=repo_path, check=True
        )

        # Enable sparse checkout
        subprocess.run(
            ["git", "config", "core.sparseCheckout", "true"], cwd=repo_path, check=True
        )

        # Create sparse-checkout file to exclude the problematic directory
        sparse_checkout_path = os.path.join(
            repo_path, ".git", "info", "sparse-checkout"
        )
        with open(sparse_checkout_path, "w") as f:
            f.write("/*\n")
            f.write("!tests/assets/Icon*\n")  # Exclude the problematic Icon file

        # Fetch and checkout
        subprocess.run(["git", "fetch", "origin", "main"], cwd=repo_path, check=True)
        subprocess.run(["git", "checkout", "main"], cwd=repo_path, check=True)

        return True
    except Exception as e:
        print(f"Failed to clone {url} with sparse checkout: {str(e)}")
        return False


# Keep track of successful and failed repositories
successful_repos = []
failed_repos = []
repo_commit_info = {}  # Store repo info with commit hashes

# Try to set Git config to allow special characters
try:
    subprocess.run(["git", "config", "--global", "core.protectNTFS", "false"])
except:
    print("Warning: Could not set git config core.protectNTFS")

# Process each repository
for url in df_harness['repo'].to_list():
    if not url:  # Skip empty lines
        continue

    print(f"\nCloning: {url}")

    # Create a clean repo name from the URL
    repo_name = ("__").join(url.split("/")[3:])
    if repo_name.endswith(".git"):
        repo_name = repo_name[:-4]  # Remove .git extension if present

    repo_path = os.path.join(path_repo, repo_name)

    # Check if repo already exists and get its commit hash
    if os.path.exists(repo_path):
        print(f"Repository already exists at {repo_path}")
        repo_commit_info[url.split("/")[-1]] = get_commit_info(repo_path)
        print("Skipping clone...")
        continue

    # Handle the problematic Quantus repository with a special approach
    if "understandable-machine-intelligence-lab/Quantus" in url:
        print(f"Using special handling for known problematic repository: {url}")
        if clone_problematic_quantus(repo_path, url):
            successful_repos.append(url)
            repo_commit_info[url.split("/")[-1]] = get_commit_info(repo_path)
        else:
            failed_repos.append((url, "Failed with special handling"))
            safe_remove_directory(repo_path)
        continue

    try:
        result = subprocess.run(
            ["git", "clone", url, repo_path],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
        )

        if result.returncode != 0:
            print(f"Error cloning {url}: {result.stderr}")
            failed_repos.append((url, result.stderr))
            time.sleep(2)
            safe_remove_directory(repo_path)
            continue

        successful_repos.append(url)

    except Exception as e:
        print(f"Unexpected error cloning {url}: {str(e)}")
        failed_repos.append((url, str(e)))
        time.sleep(2)
        safe_remove_directory(repo_path)


Cloning: https://github.com/tatsu-lab/alpaca_eval
Repository already exists at data/repos\tatsu-lab__alpaca_eval
Skipping clone...

Cloning: https://github.com/erikbern/ann-benchmarks
Repository already exists at data/repos\erikbern__ann-benchmarks
Skipping clone...

Cloning: https://github.com/stanford-futuredata/ARES
Repository already exists at data/repos\stanford-futuredata__ARES
Skipping clone...

Cloning: https://github.com/Marker-Inc-Korea/AutoRAG
Repository already exists at data/repos\Marker-Inc-Korea__AutoRAG
Skipping clone...

Cloning: https://github.com/beir-cellar/beir
Repository already exists at data/repos\beir-cellar__beir
Skipping clone...

Cloning: https://github.com/RobustNLP/CipherChat
Repository already exists at data/repos\RobustNLP__CipherChat
Skipping clone...

Cloning: https://github.com/bigcode-project/bigcode-evaluation-harness
Repository already exists at data/repos\bigcode-project__bigcode-evaluation-harness
Skipping clone...

Cloning: https://github.com/U

In [30]:
import pandas as pd
import subprocess
import ast
import re
import os
from pathlib import Path

# Base directory containing all repositories
path_repo = Path("data/repos")

# These prefixes are used to identify internal or private files that should be excluded from analysis
internal_usage_prefix = [
    "_",
    ".",
]

# Keywords to exclude during analysis
test_keywords = [
    "test"
]
usecase_keywords = [
    "debug",
    "demo",
    "example",
    "experiment",
    "notebook",
    "result",
    "tutorial",
]
deprecated_keywords = [
    "deprecated",
    "deprecation",
    "legacy",
    "obsolete",
    "old",
    "retired",
]
usecase_test_keywords = test_keywords + usecase_keywords
test_deprecated_keywords = test_keywords + deprecated_keywords
excluded_keywords = test_keywords + usecase_keywords + deprecated_keywords

# Prepare the --exclude argument for deadcode
exclude_arg = ",".join([f"*{dir}*" for dir in usecase_test_keywords])

# Dictionary to store all results
evaluation_results = {}


def preprocess_path(path):
    path_parts = path.split(os.sep)
    org_name, repo_name = path_parts[2].split("__")[0], path_parts[2].split("__")[1]
    file_path = os.path.join(*path_parts[3:]).replace(os.sep, "/")
    file_path_url = f"https://github.com/{org_name}/{repo_name}/blob/{repo_commit_info[repo_name]}/{file_path}"
    return file_path_url


def find_cli_commands(repo_path):
    """Find all CLI calls in the repository."""
    results = []

    for root, dirs, files in os.walk(repo_path):
        for file in files:
            filepath = os.path.join(root, file)
            
            has_internal_usage = False
            for prefix in internal_usage_prefix:
                for subdir in filepath.split(os.sep):
                    if subdir.startswith(prefix):
                        has_internal_usage = True
                        break
                if has_internal_usage:
                    break

            has_forbidden_usage = False
            for keyword in excluded_keywords:
                if keyword in filepath.lower():
                    has_forbidden_usage = True
                    break

            if has_internal_usage or has_forbidden_usage or not file.endswith(".py"):
                continue
        
            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()
                    file_path_url = preprocess_path(filepath)
                    for line_num, line in enumerate(content.split("\n"), 1):
                        # Look for the main entry point
                        matches = re.findall(
                            r"^\s*(?!#+)if\s+__name__\s*==\s*(['\"])(__main__)\1\s*:",
                            line,
                            re.MULTILINE,
                        )
                        if matches:
                            cli_dict = {
                                "type": "cli",
                                "line of code": f"{file_path_url}#L{line_num}",
                                "content": content.strip(),
                            }
                            results.append(cli_dict)

            except Exception as e:
                print(f"Error processing {filepath}: {e}")
                continue

    return results


def find_typer_commands(repo_path):
    """Find all typer command/option/argument decorators in the repository."""
    results = []
    repo_name = "__".join(str(repo_path.as_posix()).split("/")[-2:])

    for root, dirs, files in os.walk(repo_path):
        for file in files:
            filepath = os.path.join(root, file)
            
            has_internal_usage = False
            for prefix in internal_usage_prefix:
                for subdir in filepath.split(os.sep):
                    if subdir.startswith(prefix):
                        has_internal_usage = True
                        break
                if has_internal_usage:
                    break

            has_forbidden_usage = False
            for keyword in excluded_keywords:
                if keyword in filepath.lower():
                    has_forbidden_usage = True
                    break

            if has_internal_usage or has_forbidden_usage or not file.endswith(".py"):
                continue

            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()

                if re.search(r'^\s*import\s+typer|^\s*from\s+[\.]*typer(\.[\w]*)*\s+import', content):
                    file_path_url = preprocess_path(filepath)
                    for line_num, line in enumerate(content.split("\n"), 1):
                        if ".command(" in line:
                            # Look for the command in this line first
                            matches = re.findall(
                                r"^\s*@[\w]+\.command\(.*?\)",
                                line,
                                re.MULTILINE,
                            )
                            if matches:
                                function_name = None
                                content_lines = content.split("\n")
                                
                                # Look for the function definition in the next few lines
                                for i in range(line_num, len(content_lines)):
                                    next_line = content_lines[i].strip()
                                    func_match = re.match(r'def\s+(\w+)\s*\(', next_line)
                                    if func_match:
                                        function_name = func_match.group(1)
                                        break
                                
                                # Skip if no function name found
                                if not function_name:
                                    continue

                                has_test_deprecated_usage = False
                                for keyword in test_deprecated_keywords:
                                    if keyword in function_name.lower():
                                        has_test_deprecated_usage = True
                                        break
            
                                has_internal_usage = False
                                for prefix in internal_usage_prefix:
                                    if function_name.startswith(prefix):
                                        has_internal_usage = True
                                        break

                                if has_test_deprecated_usage or has_internal_usage:
                                    continue

                                repo_function = f"{repo_name}/{function_name}"
                                click_dict = {
                                    "type": "click",
                                    "line of code": f"{file_path_url}#L{line_num}",
                                    "component/function/instruction": repo_function,
                                    "content": content.strip(),
                                }
                                results.append(click_dict)

            except Exception as e:
                print(f"Error processing {filepath}: {e}")
                continue

    return results


def find_click_commands(repo_path):
    """Find all click command/option/argument decorators in the repository."""
    results = []
    repo_name = "__".join(str(repo_path.as_posix()).split("/")[-2:])

    for root, dirs, files in os.walk(repo_path):
        for file in files:
            filepath = os.path.join(root, file)
            
            has_internal_usage = False
            for prefix in internal_usage_prefix:
                for subdir in filepath.split(os.sep):
                    if subdir.startswith(prefix):
                        has_internal_usage = True
                        break
                if has_internal_usage:
                    break

            has_forbidden_usage = False
            for keyword in excluded_keywords:
                if keyword in filepath.lower():
                    has_forbidden_usage = True
                    break

            if has_internal_usage or has_forbidden_usage or not file.endswith(".py"):
                continue

            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()

                if re.search(r'^\s*import\s+click|^\s*from\s+[\.]*click(\.[\w]*)*\s+import', content):
                    file_path_url = preprocess_path(filepath)
                    for line_num, line in enumerate(content.split("\n"), 1):
                        if ".add_command(" in line:
                            # Look for the command in this line first
                            matches = re.findall(
                                r"^\s*([\w]+)\.add_command\(([\w]+)((,|\().+)?\)",
                                line,
                                re.MULTILINE,
                            )
                            if matches:
                                function_name = matches[0][1]

                                has_test_deprecated_usage = False
                                for keyword in test_deprecated_keywords:
                                    if keyword in function_name.lower():
                                        has_test_deprecated_usage = True
                                        break
            
                                has_internal_usage = False
                                for prefix in internal_usage_prefix:
                                    if function_name.startswith(prefix):
                                        has_internal_usage = True
                                        break

                                if has_test_deprecated_usage or has_internal_usage:
                                    continue

                                repo_function = f"{repo_name}/{function_name}"
                                click_dict = {
                                    "type": "click",
                                    "line of code": f"{file_path_url}#L{line_num}",
                                    "component/function/instruction": repo_function,
                                    "content": content.strip(),
                                }
                                results.append(click_dict)

            except Exception as e:
                print(f"Error processing {filepath}: {e}")
                continue

    return results


def find_fire_commands(repo_path):
    """Find all Python-Fire CLI usages in the repository."""
    results = []
    repo_name = "__".join(str(repo_path.as_posix()).split("/")[-2:])

    for root, dirs, files in os.walk(repo_path):
        for file in files:
            filepath = os.path.join(root, file)
            
            has_internal_usage = False
            for prefix in internal_usage_prefix:
                for subdir in filepath.split(os.sep):
                    if subdir.startswith(prefix):
                        has_internal_usage = True
                        break
                if has_internal_usage:
                    break

            has_forbidden_usage = False
            for keyword in excluded_keywords:
                if keyword in filepath.lower():
                    has_forbidden_usage = True
                    break

            if has_internal_usage or has_forbidden_usage or not file.endswith(".py"):
                continue

            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    content = f.read()

                if re.search(r'^\s*import\s+fire|^\s*from\s+[\.]*fire(\.[\w]*)*\s+import', content):
                    file_path_url = preprocess_path(filepath)
                    for line_num, line in enumerate(content.split("\n"), 1):
                        matches = re.findall(
                            r"^\s*(?!#+)(fire\.)?Fire\(([\w]+?)\)",
                            line,
                            re.MULTILINE,
                        )
                        if matches:
                            function_name = matches[0][1]

                            has_test_deprecated_usage = False
                            for keyword in test_deprecated_keywords:
                                if keyword in function_name.lower():
                                    has_test_deprecated_usage = True
                                    break
            
                            has_internal_usage = False
                            for prefix in internal_usage_prefix:
                                if function_name.startswith(prefix):
                                    has_internal_usage = True
                                    break

                            if has_test_deprecated_usage or has_internal_usage:
                                continue

                            repo_function = f"{repo_name}/{function_name}"
                            api_dict = {
                                "type": "fire",
                                "line of code": f"{file_path_url}#L{line_num}",
                                "component/function/instruction": repo_function,
                                "content": content.strip(),
                            }
                            results.append(api_dict)
            except Exception as e:
                print(f"Error processing {filepath}: {e}")
                continue

    return results


def retrieve_function(loc, function_name):
    parts = loc.split("#L")[0].split("/")
    repo_name = "__".join(parts[3:5])
    local_path = str(path_repo) + "/" + repo_name + "/" + "/".join(parts[7:])
    repo_function = repo_name + "/" + function_name

    with open(local_path, "r", encoding="utf-8") as file:
        content = file.read()

    # Extract the specific line number if needed
    # Parse the content into an AST
    tree = ast.parse(content)

    # Find the function by name
    for node in ast.walk(tree):
        if (
            isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
            and node.name == function_name
        ):
            # Get the line numbers for the function
            start_line = node.lineno - 1  # Convert to 0-based indexing
            end_line = (
                node.end_lineno if hasattr(node, "end_lineno") else start_line + 1
            )

            # Extract the function code
            lines = content.split("\n")
            function_content = "\n".join(lines[start_line:end_line])
            break

    return repo_function, function_content.strip()


df = []
path_repo = Path("data/repos")

for repo_path in path_repo.iterdir():
    if ".json" in str(repo_path):
        continue  # Skip JSON files

    result = subprocess.run(
        ["vulture", str(repo_path), "--exclude", exclude_arg, "--min-confidence", "60"],
        capture_output=True,
        text=True
    )
    
    repo_dict = []
    # unique_code_lines = set()
    for line in result.stdout.splitlines():
        if " function " in line or " method " in line:
            
            has_internal_usage = False
            for prefix in internal_usage_prefix:
                for subdir in line.split(os.sep):
                    if subdir.startswith(prefix):
                        has_internal_usage = True
                        break
                if has_internal_usage:
                    break

            has_forbidden_usage = False
            for keyword in excluded_keywords:
                if keyword in line.lower():
                    has_forbidden_usage = True
                    break
                
            if not has_internal_usage and not has_forbidden_usage:
                loc = preprocess_path(line.split(": ")[0])
                loc = re.sub(r"\.py:(\d+)", r".py#L\1", loc)
          
                function_name = line.split("'")[1]
                has_test_deprecated_usage = False
                for keyword in test_deprecated_keywords:
                    if keyword in function_name.lower():
                        has_test_deprecated_usage = True
                        break
            
                has_internal_usage = False
                for prefix in internal_usage_prefix:
                    if function_name.startswith(prefix):
                        has_internal_usage = True
                        break

                if has_test_deprecated_usage or has_internal_usage:
                    continue

                repo_function, content = retrieve_function(loc, function_name)
                api_dict = {
                    "type": "api",
                    "line of code": loc.split(": ")[0],
                    "component/function/instruction": repo_function,
                    "content": content,
                }
                repo_dict.append(api_dict)

    # Store unused function/method lines
    repo_dict.extend(find_cli_commands(repo_path))
    repo_dict.extend(find_click_commands(repo_path))
    
    # Removing the "fire" type from the dataset as it represents CLI calls identical to __main__ calls, making them redundant.
    # repo_dict.extend(find_fire_commands(repo_path))

    if not repo_dict:
        print(f"No unused code found in {repo_path.name}")
        continue

    df.extend(repo_dict)

# Write all results to a single JSON file
df = pd.DataFrame(df)
df.to_json("data/unused_python_functions.jsonl", orient="records", lines=True)

No unused code found in dezoito__ollama-grid-search


In [13]:
import pandas as pd
import plotly.express as px

df = pd.read_json("data/unused_python_functions.jsonl", lines=True)
df["filepath"] = df["line of code"].apply(lambda x: x.split("#L")[0])
print(f"Total number of unused functions/scripts: {len(df)}")
print(
    f"Total number of files with at least one unused function: {df['filepath'].nunique()}"
)

# Extract the file name for each line of code and count how many lines per file
file_counts = df.groupby("filepath").size().reset_index(name="lines_per_file")

# Create distribution data: count how many files have each number of lines
distribution = file_counts["lines_per_file"].value_counts().reset_index()
distribution.columns = ["lines_of_code", "number_of_files"]
distribution = distribution.sort_values("lines_of_code")

# Plotting the distribution of files by number of lines of code
fig = px.bar(
    distribution,
    x="lines_of_code",
    y="number_of_files",
    labels={
        "lines_of_code": "Unused Functions/Scripts per File",
        "number_of_files": "Number of Files",
    },
    log_y=True,
)
fig.update_layout(
    margin=dict(l=20, r=20, t=40, b=20),
    xaxis=dict(title_font_size=20),
    yaxis=dict(title_font_size=20),
)
fig.show()
df['type'].value_counts()

Total number of unused functions/scripts: 5114
Total number of files with at least one unused function: 2321


type
api      4165
cli       938
click      11
Name: count, dtype: int64

In [ ]:
import pandas as pd

# Sample 358 files for representative file-level analysis
# Sample size calculation URL: https://www.calculator.net/sample-size-calculator.html?type=1&cl=95&ci=5&pp=50&ps=5114&x=Calculate
df = pd.read_json("data/unused_python_functions.jsonl", lines=True)
df["filepath"] = df["line of code"].apply(lambda x: x.split("#L")[0])
df.drop_duplicates(subset=["filepath"], inplace=True)
df_sampled = df.sample(n=358, random_state=42).reset_index(drop=True)
df_sampled.sort_values(by="line of code", inplace=True)
df_sampled.drop(
    columns=["filepath", "content"], inplace=True
)  # Remove filepath and content columns as they are not needed for crowdsourced labeling
df_sampled.to_csv(
    "data/unused_python_function_sample.csv", encoding="utf-8", index=False
)

In [17]:
# import pandas as pd

# df = pd.read_csv(
#     r"C:\Users\zhimi\Downloads\Evaluation Harnesses - Labels.csv", encoding="utf-8"
# )
# loc_label_mapping = dict(zip(df["line of code"], df["label"]))

# df = pd.read_json("data/unused_python_functions.jsonl", lines=True)
# df["label"] = df["line of code"].map(loc_label_mapping)

# df.drop(columns=["content"], inplace=True)  # Remove content column as it's not needed anymore
# df.to_csv("data/unused_python_functions_labels.csv", encoding="utf-8", index=False)

In [ ]:
# import pandas as pd

# df = pd.read_csv(r"C:\Users\zhimi\Downloads\Evaluation Harnesses - Labels.csv")
# loc_label_mapping = dict(zip(df["line of code"], df["label"]))

# df_old_sample = pd.read_csv(r"C:\Users\zhimi\Downloads\Evaluation Harnesses - Samples.csv")
# for id, row in df_old_sample.iterrows():
#     if row["line of code"] in loc_label_mapping:
#         df_old_sample.at[id, "label"] = loc_label_mapping[row["line of code"]]
#     else:
#         df_old_sample.drop(index=id, inplace=True)
#         print(row["crowdsourced rater's name"])

# df = df[~df["line of code"].isin(df_old_sample["line of code"])]

# number = 358 - len(df_old_sample)
# df_new_sample = df.sample(n=number, random_state=42)
# df_sample = pd.concat([df_old_sample, df_new_sample], ignore_index=True)
# df_sample.to_csv(
#     r"data\unused_python_function_sample_labels.csv", index=False
# )

# df_sample = df_sample[["type", "line of code", "component/function/instruction"]]
# df_sample.to_csv(
#     r"data\unused_python_function_sample.csv", index=False
# )

In [46]:
import pandas as pd
import plotly.express as px

# Load the dataset
df = pd.read_csv(r"data/unused_python_functions_labels.csv")
print(f"Total number of deprecated functions: {len(df[df['label'] == 'deprecated function'])} ({len(df[df['label'] == 'deprecated function'])/len(df)*100:.2f}%)")
print(f"Total number of passthrough functions: {len(df[df['label'] == 'passthrough function'])} ({len(df[df['label'] == 'passthrough function'])/len(df)*100:.2f}%)")
print(f"Total number of redundant functions: {len(df[df['label'] == 'redundant function'])} ({len(df[df['label'] == 'unimplemented function'])/len(df)*100:.2f}%)")
print(f"Total number of unimplemented functions: {len(df[df['label'] == 'unimplemented function'])} ({len(df[df['label'] == 'unimplemented function'])/len(df)*100:.2f}%)")
# Filter out deprecated, passthrough or unimplemented functions
df = df[~df["label"].isin(["deprecated function", "passthrough function", "redundant function", "unimplemented function"])]
print(f"Total number of API functions: {len(df[df['type'] == 'api'])} ({len(df[df['type'] == 'api'])/len(df)*100:.2f}%)")
print(f"Total number of CLI instructions: {len(df[df['type'].isin(['cli', 'click', 'fire', 'typer'])])} ({len(df[df['type'].isin(['cli', 'click', 'fire', 'typer'])])/len(df)*100:.2f}%)")
print(f"Total number of GUI components: {len(df[df['type'] == 'gui'])} ({len(df[df['type'] == 'gui'])/len(df)*100:.2f}%)")
print(f"Total number of unused functions designed for end users after filtering: {len(df)}")
print(f"Total number of unique labels: {df['label'].nunique()}")
# First, let's see what columns we have after groupby
occurrence_frequency = df.groupby("label")["type"].count().reset_index()
occurrence_frequency.rename(columns={"type": "occurrence_count"}, inplace=True)
occurrence_frequency.sort_values(by="occurrence_count", ascending=False, inplace=True)
occurrence_frequency.head(10)

Total number of deprecated functions: 13 (0.25%)
Total number of passthrough functions: 22 (0.43%)
Total number of redundant functions: 0 (6.83%)
Total number of unimplemented functions: 350 (6.83%)
Total number of API functions: 3787 (79.86%)
Total number of CLI instructions: 942 (19.87%)
Total number of GUI components: 13 (0.27%)
Total number of unused functions designed for end users after filtering: 4742
Total number of unique labels: 2815


,label,occurrence_count
1579,generate run specification,63
1304,generate action,50
1546,generate question prompt,45
137,calculate accuracy,43
805,create run specification,39
2374,return items,39
1530,generate prompt,31
1639,generate translation prompt,30
1753,install requirements,30
621,convert image to rgb,24


In [47]:
import pandas as pd
import plotly.express as px

# Load the dataset
df = pd.read_csv(r"data/unused_python_functions_labels.csv")
df = df[~df["label"].isin(["deprecated function", "passthrough function", "redundant function", "unimplemented function"])]

# First, let's see what columns we have after groupby
occurrence_frequency = df.groupby("label")["type"].count().reset_index()

# Import Counter
from collections import Counter

frequency_counts = Counter(occurrence_frequency["type"])

# Convert Counter to DataFrame for plotting
frequency_df = pd.DataFrame.from_dict(
    frequency_counts, orient="index", columns=["count"]
).reset_index()
frequency_df.columns = ["label_occurrence_frequency", "count"]

# Create bar chart for occurrence frequency
fig = px.bar(
    frequency_df,
    x="label_occurrence_frequency",
    y="count",
    title="Distribution of Label Occurrence Frequency",
    text="count",
    log_x=True,
    log_y=True,
)

fig.update_traces(textposition="outside")
fig.update_layout(xaxis_tickangle=-45, margin=dict(l=20, r=20, t=40, b=60))

fig.show()

In [54]:
import pandas as pd
import spacy
import re

from collections import defaultdict
from typing import List, Dict, Tuple, Set, Optional
from spacy.language import Language
from spacy.tokens import Doc, Span
from spacy.util import filter_spans

@Language.component("merge_hyphenated")
def merge_hyphenated(doc):
    spans_to_merge = []
    
    # Find all hyphenated patterns
    i = 0
    while i < len(doc) - 2:
        if (doc[i+1].text == "-" and 
            doc[i].is_alpha and 
            doc[i+2].is_alpha):
            # Create a span for the three tokens (word-word)
            span = Span(doc, i, i+3)
            spans_to_merge.append(span)
            i += 3  # Skip past this group to avoid overlaps
        else:
            i += 1
    
    # Filter overlapping spans and merge
    spans_to_merge = filter_spans(spans_to_merge)
    
    with doc.retokenize() as retokenizer:
        for span in spans_to_merge:
            retokenizer.merge(span)
    
    return doc

class VerbNounMapper:
    def __init__(self):
        # Load spaCy model
        try:
            self.nlp = spacy.load("en_core_web_sm")
            self.nlp.add_pipe("merge_hyphenated", before="tagger")
        except OSError:
            print("spaCy English model not found.")
            print("Please install it using:")
            print("pip install spacy")
            print("python -m spacy download en_core_web_sm")
            raise

    def split_compound_labels(self, label: str) -> List[str]:
        """
        Split compound labels into individual labels based on connectives.
        Handles:
        - "verb1 and/or verb2 noun" => ["verb1 noun", "verb2 noun"]
        - "verb noun1 and/or noun2" => ["verb noun1", "verb noun2"]
        - "verb1 noun1 and/or verb2 noun2" => ["verb1 noun1", "verb2 noun2"]
        """
        doc = self.nlp(label)
        
        for id, token in enumerate(doc):
            if token.pos_ == "CCONJ":
                if id == 1:
                    label1 = doc[id - 1].text + " " + " ".join(t.text for t in doc[id + 2:])
                    label2 = doc[id + 1].text + " " + " ".join(t.text for t in doc[id + 2:])
                    # print("scenario 1")
                    return [label1, label2]
                elif doc[id - 1].pos_ == "NOUN" and doc[id + 1].pos_ != "VERB":
                    label1 = " ".join(t.text for t in doc[:id])
                    label2 = doc[0].text + " " + " ".join(t.text for t in doc[id + 1:])
                    # print("scenario 2")
                    return [label1, label2]
                elif doc[id - 1].pos_ == "NOUN" and doc[id + 1].pos_ == "VERB":
                    label1 = " ".join(t.text for t in doc[:id])
                    label2 = " ".join(t.text for t in doc[id + 1:])
                    # print("scenario 3")
                    return [label1, label2]
        
        return [label]  # If no conjunctions found, return the original label

    def extract_verb_noun_parts(self, label: str) -> Tuple[str, str]:
        """
        Extract verb and noun parts from a simple label.
        Handles:
        1. "verb noun" - load model
        2. "verb adj noun" - load pretrained model
        2. "verb noun adp noun" - load model on cloud
        """
        parts = label.split(" ")
        verb = parts[0]  # Assume the first part is the verb
        noun = " ".join(parts[1:])
        return verb, noun

    def process_single_label(self, label: str) -> List[Tuple[str, str, str]]:
        """Process a single label and return list of (label, verb, noun) tuples"""
        # First, split compound labels if necessary
        split_labels = self.split_compound_labels(label)
        
        results = []
        for split_label in split_labels:
            verb, noun = self.extract_verb_noun_parts(split_label)
            results.append((verb, noun))
        
        return results

    def process_labels(self, df: pd.DataFrame) -> pd.DataFrame:
        """Process all labels and return DataFrame with columns: line of code, label, verb, noun"""
        results = []

        for i, row in df.iterrows():
            for verb, noun in self.process_single_label(row["label"]):
                results.append({
                    "line of code": row["line of code"],
                    "label": row["label"],
                    "verb": verb,
                    "noun": noun
                })

        return pd.DataFrame(results)

    def save_results(self, df: pd.DataFrame, filename: str):
        """Save results DataFrame to jsonl file"""
        df.to_json(filename, orient="records", lines=True)
        
    def analyze_results(self, df: pd.DataFrame):
        """Print analysis of parsing results"""
        print(f"Unique verbs: {df['verb'].nunique()}")
        print(f"Unique nouns: {df['noun'].nunique()}")
        print(f"Verb list: {df['verb'].unique()}")

# Load the dataset
df = pd.read_csv(r"data/unused_python_functions_labels.csv", encoding="utf-8")

# Filter out deprecated, passthrough or unimplemented functions
df = df[~df["label"].isin(["deprecated function", "passthrough function", "redundant function", "unimplemented function"])]

# Initialize the mapper
mapper = VerbNounMapper()

# Process labels and get DataFrame
df = mapper.process_labels(df)

# Analyze and display results
mapper.analyze_results(df)

# Save to JSONL file
mapper.save_results(df, "data/unused_python_functions_analyzed.jsonl")

Unique verbs: 270
Unique nouns: 2151
Verb list: ['accept' 'activate' 'adapt' 'add' 'adjust' 'adjusts' 'aggregate' 'align'
 'analyze' 'annotate' 'append' 'apply' 'archive' 'arrange' 'assemble'
 'assess' 'assign' 'assigns' 'attach' 'augment' 'automate' 'await'
 'benchmark' 'build' 'builds' 'loads' 'cache' 'calculate' 'clip' 'display'
 'reorganize' 'report' 'return' 'save' 'generate' 'cancel' 'capture'
 'categorize' 'check' 'record' 'checkout' 'choose' 'chunk' 'classify'
 'clean' 'format' 'prepare' 'summarize' 'trim' 'clear' 'clone' 'close'
 'collect' 'convert' 'combine' 'compare' 'compile' 'run' 'compress'
 'compute' 'concatenate' 'configure' 'consolidate' 'construct'
 'constructs' 'continue' 'copy' 'correct' 'corrects' 'count' 'crawl'
 'create' 'initialize' 'store' 'upload' 'crop' 'resize' 'de-normalize'
 'decode' 'open' 'decompress' 'decrypt' 'define' 'delete' 'deserialize'
 'destroy' 'detect' 'detects' 'detokenize' 'disable' 'discretize'
 'dispatch' 'divide' 'download' 'extract' 'load

In [55]:
verb_clusters = {
    "Data Creation & Generation": {
        "description": "Operations that create new data, content, or assets from scratch through generative processes",
        "labels": ["create", "generate", "build", "builds", "construct", "constructs", "synthesize", "establish", "define", "introduce", "assemble", "spawn"]
    },
    
    "Data Retrieval & Access": {
        "description": "Operations that fetch, load, or access existing data and resources",
        "labels": ["retrieve", "get", "fetch", "load", "loads", "read", "download", "pull", "find", "finds", "locate", "search", "query", "collect", "gather", "import", "open", "crawl", "return"]
    },
    
    "Data Processing & Transformation": {
        "description": "Operations that modify, process, or transform data from one form to another",
        "labels": ["convert", "transform", "process", "parse", "encode", "decode", "format", "formats", "serialize", "deserialize", "compress", "compresses", "decompress", "hash", "encrypt", "decrypt", "transliterate", "postprocess", "prepare", "apply", "simplify"]
    },
    
    "Data Storage & Persistence": {
        "description": "Operations that save, store, or persist data and state",
        "labels": ["save", "saves", "store", "cache", "record", "persist", "archive", "export", "write", "insert", "append", "prepend", "upload", "push", "flush", "send"]
    },
    
    "Data Manipulation & Modification": {
        "description": "Operations that alter, modify, or manipulate existing data structures",
        "labels": ["update", "adjust", "adjusts", "replace", "correct", "corrects", "fix", "adapt", "refine", "rewrite", "patch", "augment", "expand", "rename", "remix"]
    },
    
    "Data Organization & Structure": {
        "description": "Operations that organize, arrange, or restructure data and collections",
        "labels": ["sort", "group", "organize", "arrange", "reorganize", "reorder", "merge", "combine", "fuse", "consolidate", "aggregate", "split", "separate", "divide", "chunk", "flatten", "reshape", "resize", "interleave", "shuffle", "reverse"]
    },
    
    "Data Analysis & Statistical Computing": {
        "description": "Operations that perform statistical analysis, computation, and mathematical evaluation on data",
        "labels": ["compute", "calculate", "evaluate", "evaluates", "analyze", "measure", "compare", "benchmark", "test", "estimate", "score", "grade", "summarize", "report", "assess"]
    },
    
    "Data Selection & Filtering": {
        "description": "Operations that select, filter, or choose specific data subsets",
        "labels": ["select", "selects", "filter", "choose", "extract", "extracts", "sample", "locate", "exclude", "crop", "clip", "trim", "truncate"]
    },
    
    "Data Assignment & Configuration": {
        "description": "Operations that assign values, configure settings, or set parameters",
        "labels": ["set", "assign", "assigns", "configure", "setup", "initialize", "initiate", "reinitialize", "enable", "disable", "activate", "toggle", "switch", "seed", "mask", "reset", "issue"]
    },
    
    "Data Removal & Cleanup": {
        "description": "Operations that remove, delete, or clean up data and resources",
        "labels": ["delete", "remove", "drop", "clear", "clean", "sanitize", "ignore", "suppress", "revoke", "unzip", "destroy", "cancel", "terminate", "end", "finalize", "redact"]
    },
    
    "System Control & Execution": {
        "description": "Operations that control system behavior, execution flow, or process management",
        "labels": ["run", "execute", "start", "launch", "trigger", "invoke", "perform", "continue", "pause", "stop", "shutdown", "await", "yield", "dispatch", "handle", "manage", "close"]
    },
    
    "State Management & Tracking": {
        "description": "Operations that manage state, track changes, or monitor system behavior",
        "labels": ["track", "trace", "log", "register", "count", "increment", "lock", "instrument", "resolve", "unfreeze"]
    },
    
    "User Interface & Visualization": {
        "description": "Operations related to user interaction, display, and data presentation",
        "labels": ["display", "print", "serve", "share", "accept", "submit", "login", "type"]
    },
    
    "Data Duplication & Transfer": {
        "description": "Operations that copy, duplicate, or transfer data between locations",
        "labels": ["copy", "duplicate", "clone", "transfer", "move", "stream", "redirect", "checkout", "reinstall", "restore", "recover", "replay", "attach"]
    },
    
    "Data Quality & Validation": {
        "description": "Operations that ensure data quality, correctness, and system validation",
        "labels": ["validate", "verify", "check", "inspect", "profile", "enforce", "restrict"]
    },
    
    "Mathematical & Statistical Operations": {
        "description": "Operations that perform mathematical computations and statistical analysis",
        "labels": ["add", "subtract", "divide", "round", "scale", "rescale", "quantize", "interpolate", "rotate", "shift", "align", "project", "reparameterize"]
    },
    
    "Model Training & Learning": {
        "description": "Operations related to training machine learning models and learning from data",
        "labels": ["train", "pretrain"]
    },
    
    "Model Inference & Prediction": {
        "description": "Operations that apply trained models to make predictions or inferences on new data",
        "labels": ["predict", "predicts", "infer"]
    },
    
    "Optimization & Performance": {
        "description": "Operations that optimize performance, efficiency, or system behavior",
        "labels": ["optimize", "automate"]
    },
    
    "Preprocessing & Feature Engineering": {
        "description": "Operations that prepare and engineer features from raw data for model consumption",
        "labels": ["preprocess", "precompute", "discretize", "normalize", "standardize", "tokenize", "detokenize", "pad", "de-normalize"]
    },
    
    "Pattern Recognition & Classification": {
        "description": "Operations that identify patterns, classify data, or categorize information",
        "labels": ["classify", "categorize", "identify", "detect", "detects"]
    },
    
    "Computer Vision & Image Processing": {
        "description": "Operations specific to image processing, computer vision, and visual data manipulation",
        "labels": ["draw", "plot", "render", "visualize", "annotate", "highlight", "capture"]
    },
    
    "Data Reconstruction & Recovery": {
        "description": "Operations that reconstruct, recover, or rebuild data and system states",
        "labels": ["reconstruct", "simulate"]
    },
    
    "System Integration & Connectivity": {
        "description": "Operations that integrate systems, establish connections, or enable interoperability",
        "labels": ["integrate"]
    },
    
    "Software Build & Compilation": {
        "description": "Operations related to building, compiling, and preparing software for deployment",
        "labels": ["compile", "install"]
    },
    
    "Data Enumeration & Listing": {
        "description": "Operations that list, enumerate, or provide structured access to collections",
        "labels": ["list", "map"]
    },
    
    "Exception Handling & Error Management": {
        "description": "Operations that handle exceptions, errors, and exceptional conditions",
        "labels": ["raise"]
    },
    
    "Data Formatting & Presentation": {
        "description": "Operations that format data for specific presentation or output requirements",
        "labels": ["concatenate", "wrap", "slim"]
    }
}

verb_cluster_mapping = {}
for cluster, verbs in verb_clusters.items():
    for verb in verbs['labels']:
        verb_cluster_mapping[verb] = cluster

import pandas as pd
df = pd.read_json(r"data/unused_python_functions_analyzed.jsonl", lines=True)

missing_verbs = set(df["verb"].unique()) - set(verb_cluster_mapping.keys())
print(f"Missing verbs in the mapping: {missing_verbs}")

extra_verbs = set(verb_cluster_mapping.keys()) - set(df["verb"].unique())
print(f"Extra verbs in the mapping: {extra_verbs}")

df["operation type"] = df["verb"].map(verb_cluster_mapping)
df.to_json(r"data/unused_python_functions_operations.jsonl", orient="records", lines=True)

Missing verbs in the mapping: set()
Extra verbs in the mapping: set()


In [50]:
import pandas as pd

df_harness = pd.read_csv("data/evaluation_harness_metadata.csv", encoding="utf-8")
repo_harness_mapping = dict(zip(df_harness["repo"], df_harness["harness"]))

def loc_harness_mapping(loc):
    repo = "/".join(loc.split("/")[:5])
    if repo in repo_harness_mapping:
        return repo_harness_mapping[repo]
    elif "loud-CV/evalai-cli" in repo:
        return "EvalAI"

df = pd.read_json(r"data/unused_python_functions_analyzed.jsonl", lines=True)
df['harness'] = df['line of code'].apply(lambda x: loc_harness_mapping(x))

import plotly.graph_objects as go

df["functionality"] = df["verb"].map(verb_cluster_mapping)
# Create a pivot table showing which tools have which functionalities
tool_functionality_matrix = df.groupby(['functionality', 'harness']).size().unstack(fill_value=0)

# Convert to binary (1 if tool has functionality, 0 if not)
tool_functionality_binary = (tool_functionality_matrix > 0).astype(int)

# Sort the y-axis (functionality) alphabetically
tool_functionality_binary = tool_functionality_binary.sort_index(ascending=False)

fig = go.Figure(data=go.Heatmap(
    z=tool_functionality_binary.values,
    x=tool_functionality_binary.columns,
    y=tool_functionality_binary.index,
    colorscale=[[0, 'white'], [1, 'coral']],
    showscale=False  # No colorbar at all
))

fig.update_layout(
    xaxis_tickangle=-45,
    height=600,
    margin=dict(l=0, r=0, t=0, b=0),
    xaxis_title="Evaluation Harness",
    yaxis_title="Functionality",
    xaxis=dict(
        title_font_size=20,
        tickfont_size=14,
        showgrid=False,  # Turn off default grid
        side='bottom'
    ),
    yaxis=dict(
        title_font_size=20,
        tickfont_size=14,
        showgrid=False,  # Turn off default grid
        side='left'
    ),
    legend=dict(
        x=0.4,
        y=-0.3,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font_size=14
    ),
    # Add plot background color to make grid lines visible
    plot_bgcolor='white'
)

# Add manual grid lines using shapes
for i in range(len(tool_functionality_binary.index) + 1):
    fig.add_hline(y=i-0.5, line_width=1, line_color="lightgray")

for i in range(len(tool_functionality_binary.columns) + 1):
    fig.add_vline(x=i-0.5, line_width=1, line_color="lightgray")

# Add custom legend for binary values
fig.add_scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(size=30, color='coral'),
    showlegend=True, name='Externally Supported'
)
fig.add_scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(size=30, color='white', line=dict(color='black', width=0.5)),
    showlegend=True, name='Not Externally Supported'
)

fig.show()

In [ ]:
import numpy as np
import json
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings

warnings.filterwarnings("ignore")


class HybridAssetClustering:
    def __init__(self, model_name, batch_size=16):
        """
        Initialize the hybrid clustering system with Qwen3 embedding model.

        Args:
            model_name: Qwen3 embedding model to use
            batch_size: Batch size for encoding (reduced for large model)
        """
        print(f"Loading {model_name}...")
        self.model_name = model_name
        self.batch_size = batch_size
        
        # Load Qwen3 model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        
        # Set device
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.model.eval()
        
        print(f"Model loaded on {self.device}")
        
        self.seed_examples = {
            # Resources (Section 4.1.1)
            "Dataset": [
                "data",
                "dataset",
                "dataframe",
                "holdout set",
                "dev set",
                "test set",
                "test cases",
                "sample",
                "slice",
                "split",
                "corpus",
                "content",
                "document",
                "paragraph",
                "sentence",
                "summary",
                "text",
                "image",
                "audio",
                "video",
                "feature",
                "tensor",
                "csv",
                "jsonl",
                "parquet",
                "blob",
                "partition",
                "bucket",
                "chunk",
                "object",
            ],
            "Model": [
                "model",
                "checkpoint",
                "neural network",
                "decision tree",
                "random forest",
                "classifier",
                "detector",
                "learner",
                "regressor",
                "transformer",
                "llm",
                "encoder",
                "decoder",
                "autoencoder",
                "backbone",
                "bert",
                "onnx",
                "dpo",
                "flan",
                "ppo",
                "resnet",
                "lenet",
                "clip",
                "vit",
                "gan",
                "cnn",
                "lstm",
                "rnn",
                "svm",
                "llama",
                "llava",
                "chatgpt",
                "gemini",
                "mistral",
                "deepseek",
                "qwen",
                "grok",
                "claude",
                "gemma",
            ],
            # Software (Section 4.1.2)
            "Source code": [
                "program",
                "code",
                "script",
                "cli",
                "api",
                "argument",
                "command",
                "command-line",
                "function",
                "method",
                "class",
                "module",
                "submodule",
                "variable",
                "attribute",
                "python",
                "javascript",
                "rust",
                "cpp",
                "csharp",
                "dlang",
                "racket",
                "ruby",
                "repository",
                "diff",
                "pull request",
                "commit",
                "branch",
                "patch",
                "git",
            ],
            "Parameter": [
                "parameter",
                "hyperparameter",
                "learning rate",
                "batch size",
                "epoch",
                "dropout",
                "scheduler",
                "sampler",
                "optimizer",
                "gradient",
                "momentum",
                "temperature",
                "threshold",
                "layer",
                "hidden size",
                "attention head",
                "weight",
                "bias",
            ],
            "Notebook": [
                "notebook",
                "jupyter",
                "ipynb",
                "colab",
                "ipython",
            ],
            # Metadata (Section 4.1.3)
            "Experiment": [
                "experiment",
                "run",
                "trial",
            ],
            "DatasetMetadata": [
                "data schema",
                "data distribution",
                "data statistics",
                "data quality",
                "data format",
                "data provenance",
                "data license",
                "dataset size",
                "vocabulary size",
            ],
            "CodeMetadata": [
                "specification",
                "documentation",
                "docstring",
                "function signature",
                "class definition",
                "attribute name",
            ],
            "ModelMetadata": [
                "model architecture",
                "model configuration",
                "model size",
                "model version",
                "model config",
                "model card",
                "model description",
                "model license",
            ],
            # Execution Data (Section 4.1.4)
            "Environment": [
                "environment variable",
                "ec2 instance",
                "project manager",
                "configuration",
                "settings",
                "yaml",
                "os",
            ],
            "Library": [
                "library",
                "dependency",
                "installation",
                "requirements",
                "version",
                "package",
            ],
            "Container": [
                "container",
                "docker",
                "kubernetes",
                "eks node",
                "singularity",
                "containerd",
                "podman",
                "lxc",
            ],
            "Stage": [
                "phase",
                "stage",
                "scenario",
                "step",
                "milestone",
                "iteration",
            ],
            "Pipeline": [
                "pipeline",
                "workflow",
                "process",
                "orchestration",
                "procedure",
                "task",
                "job",
                "ci",
                "cd",
            ],
            "Hardware": [
                "hardware",
                "machine",
                "device",
                "server",
                "pc",
                "mac",
                "cpu",
                "network",
                "memory",
                "disk",
                "gpu",
                "tpu",
            ],
            "Execution state": [
                "executing",
                "running",
                "stopped",
                "resumed",
                "failed",
                "succeeded",
                "started",
                "completed",
                "pending",
                "queued",
                "cancelled",
                "interrupted",
                "paused",
                "timeout",
                "success",
                "error",
                "finished",
                "saved",
                "ended",
                "loaded",
            ],
            "Console log": [
                "event",
                "log",
                "message",
                "debug",
                "warning",
                "error",
                "trace",
                "print",
            ],
            "Metrics": [
                "metrics",
                "score",
                "roc",
                "auc",
                "mean",
                "std",
                "min",
                "max",
                "sum",
                "count",
                "total",
                "average",
                "result",
                "accuracy",
                "precision",
                "recall",
                "f1",
                "percentage",
                "rating",
                "ranking",
                "level",
                "micro",
                "macro",
                "loss",
                "auc",
                "rmse",
                "mae",
                "mse",
                "bleu",
                "bleurt",
                "sacrebleu",
                "rouge",
                "iou",
                "gleu",
                "cider",
                "meteor",
                "pass@",
                "win rate",
                "perplexity",
                "exact match",
                "bertscore",
                "kendall's tau"
                "meteor",
                "reward",
                "distance",
                "correlation",
                "similarity",
                "confusion matrix",
            ],
            "Plot": [
                "plot",
                "subplot",
                "visualization",
                "histogram",
                "heatmap",
                "figure",
                "graph",
                "curve",
                "bar",
                "line",
                "table",
                "chart",
                "dashboard",
                "legend",
                "color map",
                "axis",
                "grid",
                "spectrogram",
                "scatter",
                "radar",
                "pie",
            ],
            # newly added categories
            "FileMetadata": [
                "file",
                "directory",
                "folder",
                "subfolder",
                "path",
            ],
            "GroundTruth": [
                "ground truth",
                "gold patch",
                "reference",
                "annotation",
                "target",
                "label",
                "answer",
                "tag",
            ],
            "AccessMetadata": [
                "access",
                "permission",
                "authorization",
                "authentication",
                "credential",
                "login",
                "role",
                "id",
                "account",
                "group",
                "secret",
                "bearer",
                "oauth",
                "service key",
                "api key",
            ],
            "Prompt": [
                "prompt",
                "query",
                "input",
            ],
            "Embedding": [
                "embedding",
                "token",
                "subtoken",
                "vector",
                "tokenizer",
                "attention mask",
                "search index",
            ],
            "Evals": [
                "evals",
                "evaluation",
                "analysis",
                "report",
                "benchmark",
                "challenge",
                "competition",
                "leaderboard",
                "submission",
                "participant",
                "contest",
                "team",
            ],
        }

        self.labels = None
        self.embeddings = None
        self.seed_centroids = None
        # Use defaultdict(set) for unique terms
        self.clusters = defaultdict(set)
        self.cluster_assignments = None
        # Pre-compute seed term lookup for fast string matching
        self.seed_term_lookup = None

    def encode(self, sentences, max_length=512, normalize=True, show_progress=True):
        """
        Encode sentences using Qwen3 embedding model.
        
        Args:
            sentences: String or list of strings to encode
            max_length: Maximum token length
            normalize: Whether to normalize embeddings to unit vectors
            show_progress: Whether to show progress
            
        Returns:
            numpy array of embeddings
        """
        if isinstance(sentences, str):
            sentences = [sentences]
        
        all_embeddings = []
        
        # Process in batches with progress indication
        total_batches = (len(sentences) + self.batch_size - 1) // self.batch_size
        
        for i in range(0, len(sentences), self.batch_size):
            batch_sentences = sentences[i:i + self.batch_size]
            
            if show_progress and total_batches > 1:
                batch_num = i // self.batch_size + 1
                print(f"Processing batch {batch_num}/{total_batches}...")
            
            batch_embeddings = self._encode_batch(batch_sentences, max_length)
            all_embeddings.append(batch_embeddings)
        
        # Concatenate all embeddings
        embeddings = np.vstack(all_embeddings)
        
        # Normalize if requested
        if normalize:
            embeddings = F.normalize(torch.from_numpy(embeddings), p=2, dim=1).numpy()
            
        return embeddings
    
    def _encode_batch(self, sentences, max_length):
        """Encode a batch of sentences"""
        # Tokenize
        inputs = self.tokenizer(
            sentences, 
            padding=True, 
            truncation=True, 
            return_tensors='pt', 
            max_length=max_length
        )
        
        # Move to device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Get embeddings
        with torch.no_grad():
            outputs = self.model(**inputs)
            
            # Use mean pooling to get sentence embeddings
            embeddings = self._mean_pooling(outputs.last_hidden_state, inputs['attention_mask'])
            
        return embeddings.cpu().numpy()
    
    def _mean_pooling(self, token_embeddings, attention_mask):
        """Apply mean pooling to token embeddings"""
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        return sum_embeddings / sum_mask

    def _build_seed_term_lookup(self):
        """Build a lookup dictionary for fast exact string matching."""
        self.seed_term_lookup = {}
        for category, terms in self.seed_examples.items():
            for term in terms:
                # Store the exact term (case-insensitive)
                term_lower = term.lower()
                self.seed_term_lookup[term_lower] = category

    def _find_string_match_category(self, label):
        """
        Find if a label has matches with seed terms.
        Returns (category, match_type, matched_term) or (None, None, None)
        
        Match priority:
        1. EXACT match (highest priority - direct assignment)
        2. Partial matches (lower priority - needs embedding verification)
        """
        label_lower = label.lower()
        
        # 1. Check for EXACT match first (highest priority)
        if label_lower in self.seed_term_lookup:
            return self.seed_term_lookup[label_lower], 'exact', label_lower
        
        # 2. Check for partial matches (need embedding verification)
        label_normalized = label_lower.replace('_', ' ').replace('-', ' ')
        label_words = label_normalized.split()
        
        # Check if any word in the label exactly matches a seed term
        for word in label_words:
            if len(word) > 2 and word in self.seed_term_lookup:  # Skip very short words
                return self.seed_term_lookup[word], 'word_exact', word
        
        # Check for substring matches (seed term is substring of label)
        for seed_term, category in self.seed_term_lookup.items():
            if len(seed_term) > 3:  # Skip very short terms to avoid false matches
                if seed_term in label_lower:
                    return category, 'substring', seed_term
        
        # Check for reverse substring (label is substring of seed term)
        for seed_term, category in self.seed_term_lookup.items():
            if len(label_lower) > 3 and label_lower in seed_term:
                return category, 'reverse_substring', seed_term
        
        return None, None, None

    def load_data(self, file_path, label_column="label"):
        """
        Load labels from JSONL file.

        Args:
            file_path: Path to JSONL file
            label_column: Column name containing the labels
        """
        self.labels = []
        with open(file_path, "r") as f:
            for line in f:
                data = json.loads(line.strip())
                if label_column in data:
                    self.labels.append(data[label_column])

        print(f"Loaded {len(self.labels)} labels from {file_path}")
        
        # Build seed term lookup after loading data
        self._build_seed_term_lookup()
        
        return self.labels

    def generate_embeddings(self):
        """Generate embeddings for all labels and seed examples using Qwen3."""
        print("Generating embeddings with Qwen3...")

        # Combine all labels and seed examples
        all_texts = self.labels.copy()
        seed_texts = []
        for asset_type, examples in self.seed_examples.items():
            seed_texts.extend(examples)

        all_texts.extend(seed_texts)

        # Generate embeddings using Qwen3
        embeddings = self.encode(all_texts, show_progress=True)

        # Split embeddings
        self.embeddings = embeddings[: len(self.labels)]
        seed_embeddings = embeddings[len(self.labels) :]

        # Calculate seed centroids
        self.seed_centroids = {}
        start_idx = 0
        for asset_type, examples in self.seed_examples.items():
            end_idx = start_idx + len(examples)
            centroid = np.mean(seed_embeddings[start_idx:end_idx], axis=0)
            self.seed_centroids[asset_type] = centroid
            start_idx = end_idx

        print(f"Generated embeddings: {self.embeddings.shape}")
        return self.embeddings

    def phase1_seed_assignment(self, confidence_threshold=0.5, relative_threshold=0.15, 
                              substring_bonus=0.3):
        """
        Refined Phase 1: Exact matches get direct assignment, partial matches need embedding verification.
        
        Args:
            confidence_threshold: Minimum similarity for non-exact matches
            relative_threshold: Minimum difference between best and second-best match
            substring_bonus: Bonus added to similarity for partial string matches
        """
        print("\nPhase 1: Seed assignment with refined string matching...")
        print(f"  Confidence threshold: {confidence_threshold}")
        print(f"  Relative threshold: {relative_threshold}")
        print(f"  Substring bonus: {substring_bonus}")

        assignments = {}
        confidences = {}
        unassigned = []
        
        # Track match types for analysis
        match_stats = {
            'exact': 0,
            'word_exact': 0,
            'substring': 0,
            'reverse_substring': 0,
            'embedding_only': 0
        }

        for i, label in enumerate(self.labels):
            label_embedding = self.embeddings[i].reshape(1, -1)
            
            # Check for string matches
            match_category, match_type, matched_term = self._find_string_match_category(label)
            
            # EXACT matches get direct assignment (no embedding verification needed)
            if match_type == 'exact':
                assignments[label] = match_category
                confidences[label] = 1.0  # Maximum confidence for exact matches
                match_stats['exact'] += 1
                continue
            
            # For partial matches and no matches, use embedding-based approach
            # Calculate embedding similarities to all seed centroids
            similarities = {}
            for asset_type, centroid in self.seed_centroids.items():
                centroid_reshaped = centroid.reshape(1, -1)
                sim = cosine_similarity(label_embedding, centroid_reshaped)[0, 0]
                
                # Apply bonus if there's a partial string match with this category
                if match_category == asset_type and match_type != 'exact':
                    sim = min(1.0, sim + substring_bonus)
                
                similarities[asset_type] = sim

            # Sort similarities to find best and second-best
            sorted_sims = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
            best_type, best_sim = sorted_sims[0]
            
            # Different thresholds for partial matches vs no matches
            if match_category:
                # Partial string match - more lenient threshold
                threshold_to_use = confidence_threshold * 0.7
                # Also prefer the string-matched category if similarity is close
                if match_category == best_type or similarities[match_category] >= threshold_to_use:
                    assignments[label] = match_category
                    confidences[label] = similarities[match_category]
                    match_stats[match_type] += 1
                    continue
            
            # No string match or string match didn't meet criteria - use best embedding match
            threshold_to_use = confidence_threshold
            
            # Check if there's a clear winner
            assign = False
            if best_sim >= threshold_to_use:
                if len(sorted_sims) > 1:
                    second_best_sim = sorted_sims[1][1]
                    if (best_sim - second_best_sim) >= relative_threshold:
                        assign = True
                else:
                    assign = True
            
            if assign:
                assignments[label] = best_type
                confidences[label] = best_sim
                if match_category and match_category == best_type:
                    match_stats[match_type] += 1
                else:
                    match_stats['embedding_only'] += 1
            else:
                unassigned.append(label)

        print(f"\nAssignment statistics:")
        print(f"  Total assigned to seed clusters: {len(assignments)}")
        print(f"  Match type breakdown:")
        for match_type, count in match_stats.items():
            if count > 0:
                print(f"    {match_type}: {count}")
        print(f"  Unassigned: {len(unassigned)}")

        return assignments, confidences, unassigned

    def phase2_discover_new_clusters(self, unassigned_labels, min_cluster_size=5, eps=0.25,
                                    skip_if_few=True, min_unassigned_for_discovery=50):
        """
        Phase 2: Discover new clusters - but be very restrictive.
        
        Args:
            unassigned_labels: Labels not assigned in phase 1
            min_cluster_size: Minimum cluster size
            eps: DBSCAN epsilon parameter
            skip_if_few: Skip discovery if too few unassigned labels
            min_unassigned_for_discovery: Minimum unassigned labels needed to run discovery
        """
        if not unassigned_labels:
            return {}
        
        # Skip new cluster discovery if we have too few unassigned labels
        if skip_if_few and len(unassigned_labels) < min_unassigned_for_discovery:
            print(f"\nPhase 2: Skipping new cluster discovery (only {len(unassigned_labels)} unassigned)")
            return {}

        print(f"\nPhase 2: Discovering new clusters from {len(unassigned_labels)} unassigned labels...")
        print(f"  Min cluster size: {min_cluster_size}")
        print(f"  DBSCAN eps: {eps}")

        # Get embeddings for unassigned labels
        unassigned_indices = [self.labels.index(label) for label in unassigned_labels]
        unassigned_embeddings = self.embeddings[unassigned_indices]

        # Try DBSCAN clustering with stricter parameters
        dbscan = DBSCAN(eps=eps, min_samples=min_cluster_size, metric="cosine")
        cluster_labels = dbscan.fit_predict(unassigned_embeddings)

        # Organize discovered clusters using sets
        discovered_clusters = defaultdict(set)
        for i, cluster_id in enumerate(cluster_labels):
            if cluster_id != -1:  # Not noise
                discovered_clusters[f"new_cluster_{cluster_id}"].add(
                    unassigned_labels[i]
                )

        # Filter out small clusters
        discovered_clusters = {
            k: v for k, v in discovered_clusters.items() if len(v) >= min_cluster_size
        }

        print(f"Discovered {len(discovered_clusters)} new clusters:")
        for cluster_name, members in discovered_clusters.items():
            sample = list(members)[:5]
            print(f"  {cluster_name}: {len(members)} members (sample: {sample}...)")

        return discovered_clusters

    def phase3_refinement(self, initial_assignments, discovered_clusters, max_iterations, min_similarity,
                          seed_bias=0.1):
        """
        Phase 3: Iterative refinement with bias toward seed clusters.
        Exact matches are protected from reassignment.
        
        Args:
            initial_assignments: Phase 1 assignments
            discovered_clusters: Phase 2 discovered clusters
            max_iterations: Number of refinement iterations
            min_similarity: Minimum similarity for assignment
            seed_bias: Additional similarity bonus for seed clusters
        """
        print(f"\nPhase 3: Refining clusters with min similarity {min_similarity}...")
        print(f"  Seed cluster bias: {seed_bias}")

        # Track which clusters are seed-based and which labels had exact matches
        seed_cluster_names = set(self.seed_examples.keys())
        exact_match_labels = set()
        
        # Identify exact match labels to protect them from reassignment
        for label in self.labels:
            match_category, match_type, _ = self._find_string_match_category(label)
            if match_type == 'exact':
                exact_match_labels.add(label)

        # Initialize clusters with sets
        all_clusters = defaultdict(set)

        # Add initial assignments
        for label, asset_type in initial_assignments.items():
            all_clusters[asset_type].add(label)

        # Add discovered clusters
        for cluster_name, members in discovered_clusters.items():
            all_clusters[cluster_name] = members.copy()

        # Iterative refinement
        for iteration in range(max_iterations):
            print(f"  Refinement iteration {iteration + 1}")

            # Recalculate centroids
            new_centroids = {}
            for cluster_name, members in all_clusters.items():
                if members:
                    member_indices = [
                        self.labels.index(label)
                        for label in members
                        if label in self.labels
                    ]
                    if member_indices:
                        cluster_embeddings = self.embeddings[member_indices]
                        new_centroids[cluster_name] = np.mean(
                            cluster_embeddings, axis=0
                        )

            # Reassign labels based on new centroids
            reassignments = 0
            new_clusters = defaultdict(set)

            for label in self.labels:
                # Skip reassignment for exact matches - they stay in their original cluster
                if label in exact_match_labels:
                    # Find current cluster for exact match labels
                    for cluster_name, members in all_clusters.items():
                        if label in members:
                            new_clusters[cluster_name].add(label)
                            break
                    continue
                
                label_embedding = self.embeddings[self.labels.index(label)].reshape(
                    1, -1
                )
                
                # Check for string match preference (but not exact)
                match_category, match_type, _ = self._find_string_match_category(label)

                # Find best cluster
                best_cluster = None
                best_similarity = -1
                similarities = []

                for cluster_name, centroid in new_centroids.items():
                    centroid_reshaped = centroid.reshape(1, -1)
                    sim = cosine_similarity(label_embedding, centroid_reshaped)[0, 0]
                    
                    # Apply bias for seed clusters
                    if cluster_name in seed_cluster_names:
                        sim = min(1.0, sim + seed_bias)
                    
                    # Extra bias if partial string match (not exact)
                    if match_category == cluster_name and match_type != 'exact':
                        sim = min(1.0, sim + 0.2)
                    
                    similarities.append((cluster_name, sim))

                    if sim > best_similarity:
                        best_similarity = sim
                        best_cluster = cluster_name

                # Sort to check relative differences
                similarities.sort(key=lambda x: x[1], reverse=True)
                
                # Relaxed criteria for partial string-matched terms
                threshold_to_use = min_similarity * 0.7 if match_category else min_similarity
                
                # Apply assignment criteria
                assign = False
                if best_cluster and best_similarity > threshold_to_use:
                    if len(similarities) > 1 and not match_category:
                        # Check separation from second-best only for non-string-matched
                        second_best_sim = similarities[1][1]
                        if (best_similarity - second_best_sim) >= 0.1:
                            assign = True
                    else:
                        assign = True
                
                if assign:
                    new_clusters[best_cluster].add(label)

                    # Check if this is a reassignment
                    old_cluster = None
                    for cluster_name, members in all_clusters.items():
                        if label in members:
                            old_cluster = cluster_name
                            break

                    if old_cluster and old_cluster != best_cluster:
                        reassignments += 1

            all_clusters = new_clusters
            print(f"    Reassignments: {reassignments} (exact matches protected)")

            if reassignments < 5:  # Convergence threshold
                break

        self.clusters = all_clusters
        return all_clusters

    def force_assign_remaining(self):
        """
        Force assign any remaining unclustered labels to their best matching cluster.
        Prioritizes seed clusters and respects exact string matches.
        """
        # Find unclustered labels
        all_clustered = set()
        for members in self.clusters.values():
            all_clustered.update(members)
        remaining = set(self.labels) - all_clustered
        
        if not remaining:
            return 0
        
        print(f"\nPhase 4: Force assigning {len(remaining)} remaining labels...")
        
        # Track seed clusters
        seed_cluster_names = set(self.seed_examples.keys())
        
        for label in remaining:
            # First check for any string match (prioritize exact matches)
            match_category, match_type, _ = self._find_string_match_category(label)
            
            if match_category and match_category in self.clusters:
                # Direct assignment based on string match
                self.clusters[match_category].add(label)
                continue
            
            # Otherwise use embedding similarity
            label_idx = self.labels.index(label)
            label_embedding = self.embeddings[label_idx].reshape(1, -1)
            
            best_cluster = None
            best_similarity = -1
            
            # Calculate current cluster centroids
            for cluster_name, members in self.clusters.items():
                if members:
                    member_indices = [self.labels.index(l) for l in members if l in self.labels]
                    cluster_embeddings = self.embeddings[member_indices]
                    centroid = np.mean(cluster_embeddings, axis=0).reshape(1, -1)
                    
                    sim = cosine_similarity(label_embedding, centroid)[0, 0]
                    
                    # Bias toward seed clusters
                    if cluster_name in seed_cluster_names:
                        sim += 0.1
                    
                    if sim > best_similarity:
                        best_similarity = sim
                        best_cluster = cluster_name
            
            if best_cluster:
                self.clusters[best_cluster].add(label)
        
        return len(remaining)

    def analyze_clusters(self):
        """Analyze and report cluster quality."""
        if not self.clusters:
            print("No clusters to analyze. Run clustering first.")
            return

        print("\n" + "=" * 50)
        print("CLUSTER ANALYSIS RESULTS")
        print("=" * 50)

        total_labels = sum(len(members) for members in self.clusters.values())
        print(f"Total labels clustered: {total_labels}/{len(self.labels)}")
        print(f"Number of clusters: {len(self.clusters)}")
        
        # Separate seed and new clusters
        seed_cluster_names = set(self.seed_examples.keys())
        seed_clusters = {k: v for k, v in self.clusters.items() if k in seed_cluster_names}
        new_clusters = {k: v for k, v in self.clusters.items() if k not in seed_cluster_names}
        
        seed_total = sum(len(members) for members in seed_clusters.values())
        new_total = sum(len(members) for members in new_clusters.values())
        
        print(f"\nCluster distribution:")
        print(f"  Seed clusters: {len(seed_clusters)} clusters, {seed_total} labels ({seed_total/len(self.labels)*100:.1f}%)")
        print(f"  New clusters: {len(new_clusters)} clusters, {new_total} labels ({new_total/len(self.labels)*100:.1f}%)")

        # Analyze string match quality for seed clusters
        print(f"\nString match analysis for seed clusters:")
        exact_matches = 0
        partial_matches = 0
        no_matches = 0
        
        for cluster_name, members in seed_clusters.items():
            for label in members:
                match_category, match_type, _ = self._find_string_match_category(label)
                if match_type == 'exact':
                    exact_matches += 1
                elif match_type:
                    partial_matches += 1
                else:
                    no_matches += 1
        
        if seed_total > 0:
            print(f"  Exact string matches: {exact_matches} ({exact_matches/seed_total*100:.1f}%)")
            print(f"  Partial string matches: {partial_matches} ({partial_matches/seed_total*100:.1f}%)")
            print(f"  Embedding-only assignments: {no_matches} ({no_matches/seed_total*100:.1f}%)")

        # Cluster size distribution
        sizes = [len(members) for members in self.clusters.values()]
        print(f"\nCluster size statistics:")
        print(f"  Mean: {np.mean(sizes):.1f}")
        print(f"  Median: {np.median(sizes):.1f}")
        print(f"  Min: {min(sizes)}, Max: {max(sizes)}")

        # Detailed cluster information
        print(f"\nDetailed cluster breakdown:")
        
        # Show seed clusters first
        if seed_clusters:
            print("\nSeed Clusters:")
            for cluster_name, members in sorted(
                seed_clusters.items(), key=lambda x: len(x[1]), reverse=True
            ):
                print(f"  {cluster_name} ({len(members)} unique labels)")
                sample_size = min(5, len(members))
                sample = sorted(list(members))[:sample_size]
                print(f"    Sample: {', '.join(sample)}")
                if len(members) > sample_size:
                    print(f"    ... and {len(members) - sample_size} more")
        
        # Show new clusters
        if new_clusters:
            print("\nNew Clusters:")
            for cluster_name, members in sorted(
                new_clusters.items(), key=lambda x: len(x[1]), reverse=True
            ):
                print(f"  {cluster_name} ({len(members)} unique labels)")
                sample_size = min(5, len(members))
                sample = sorted(list(members))[:sample_size]
                print(f"    Sample: {', '.join(sample)}")
                if len(members) > sample_size:
                    print(f"    ... and {len(members) - sample_size} more")

        return self.clusters

    def save_results(self, output_path):
        """
        Save clustering results to JSON file.
        Converts sets to sorted lists for JSON serialization.
        """
        if not self.clusters:
            print("No clusters to save. Run clustering first.")
            return

        # Convert sets to sorted lists for JSON serialization
        json_compatible_clusters = {
            cluster_name: sorted(list(members))
            for cluster_name, members in self.clusters.items()
        }

        with open(output_path, "w") as f:
            json.dump(json_compatible_clusters, f, indent=2)

        print(f"\nResults saved to {output_path}")

    def visualize_clusters(self, max_labels_per_cluster=20):
        """Create visualizations of the clustering results."""
        if not self.clusters:
            print("No clusters to visualize. Run clustering first.")
            return

        # Prepare data for visualization
        cluster_names = list(self.clusters.keys())
        cluster_sizes = [len(self.clusters[name]) for name in cluster_names]

        # Sort by size and take top 20, then aggregate the rest as "Others"
        sorted_clusters = sorted(
            zip(cluster_names, cluster_sizes), key=lambda x: x[1], reverse=True
        )

        if len(sorted_clusters) > 20:
            top_20 = sorted_clusters[:20]
            others_count = sum(size for _, size in sorted_clusters[20:])
            sorted_clusters = top_20 + [("Others", others_count)]
        else:
            sorted_clusters = sorted_clusters

        names, sizes = zip(*sorted_clusters)

        # Create the plot using seaborn
        plt.figure(figsize=(12, 8))
        ax = sns.barplot(
            x=list(sizes),
            y=[name.replace("_", " ").title() for name in names],
            palette="viridis",
        )

        # Customize the plot
        plt.xlabel("Number of Unique Labels", fontsize=12)
        plt.ylabel("Cluster Name", fontsize=12)
        plt.title(
            "Distribution of Machine Learning Asset Types (Unique Terms)",
            fontsize=14,
            fontweight="bold",
        )

        # Add value labels on bars
        for i, v in enumerate(sizes):
            ax.text(v + 0.5, i, str(v), va="center", fontsize=10)

        plt.tight_layout()
        plt.show()

    def run_full_pipeline(
        self,
        file_path,
        output_path,
        label_column,
        confidence_threshold,
        relative_threshold,
        min_cluster_size,
        dbscan_eps,
        max_iterations,
        min_similarity,
        substring_bonus=0.3,
        seed_bias=0.1,
        skip_phase2_if_few=True,
        force_complete=True
    ):
        """
        Run the complete hybrid clustering pipeline with refined string matching.
        
        Args:
            file_path: Path to input JSONL file
            output_path: Path to save results
            label_column: Column name containing labels
            confidence_threshold: Minimum similarity for non-exact matches
            relative_threshold: Minimum difference between best and second-best match
            min_cluster_size: Minimum cluster size for DBSCAN
            dbscan_eps: DBSCAN epsilon parameter
            max_iterations: Number of refinement iterations
            min_similarity: Minimum similarity for refinement
            substring_bonus: Bonus for partial string matches in embedding similarity
            seed_bias: Bias toward seed clusters in refinement
            skip_phase2_if_few: Skip new cluster discovery if few unassigned
            force_complete: Force assign remaining labels for 100% coverage
        """
        print("Starting Hybrid Asset Type Clustering Pipeline with Qwen3")
        print("=" * 60)
        print("\nConfiguration:")
        print(f"  Model: {self.model_name}")
        print(f"  Batch size: {self.batch_size}")
        print(f"  Device: {self.device}")
        print(f"  Confidence threshold: {confidence_threshold}")
        print(f"  Relative threshold: {relative_threshold}")
        print(f"  Min cluster size: {min_cluster_size}")
        print(f"  DBSCAN eps: {dbscan_eps}")
        print(f"  Refinement min similarity: {min_similarity}")
        print(f"  Substring bonus: {substring_bonus}")
        print(f"  Seed bias: {seed_bias}")
        print("=" * 60)

        # Load data (this also builds the seed term lookup)
        self.load_data(file_path, label_column)

        # Generate embeddings with Qwen3
        self.generate_embeddings()

        # Phase 1: Refined seed-based assignment with exact match priority
        assignments, confidences, unassigned = self.phase1_seed_assignment(
            confidence_threshold, relative_threshold, substring_bonus
        )

        # Phase 2: Discover new clusters (restrictive)
        discovered = self.phase2_discover_new_clusters(
            unassigned, min_cluster_size, dbscan_eps, 
            skip_if_few=skip_phase2_if_few, min_unassigned_for_discovery=50
        )

        # Phase 3: Refinement with seed bias and exact match protection
        final_clusters = self.phase3_refinement(
            assignments, discovered, max_iterations, min_similarity, seed_bias
        )
        
        # Phase 4: Force assign remaining if requested
        if force_complete:
            forced = self.force_assign_remaining()
            if forced > 0:
                print(f"  Force assigned {forced} remaining labels")

        # Analysis and output
        self.analyze_clusters()
        self.save_results(output_path)
        self.visualize_clusters()

        return final_clusters


# Example usage with Qwen3 embedding model
if __name__ == "__main__":
    # Initialize with Qwen3 model (adjust batch_size based on your GPU memory)
    clusterer = HybridAssetClustering(
        model_name="Qwen/Qwen3-Embedding-4B",
        batch_size=8  # Reduced batch size for the larger model
    )

    # Configuration that maximizes seed cluster assignment with exact match priority
    results = clusterer.run_full_pipeline(
        file_path="data/unused_python_functions_analyzed.jsonl",
        output_path="data/unused_python_functions_assets.json",
        label_column="noun",
        
        # Phase 1 parameters - refined for exact vs partial matches
        confidence_threshold=0.25,      # Threshold for embedding-only matches
        relative_threshold=0.05,        # Smaller gap needed between candidates
        
        # Phase 2 parameters - very restrictive
        min_cluster_size=10,            # Large minimum to prevent small new clusters
        dbscan_eps=0.2,                 # Tight clusters only
        
        # Phase 3 parameters - inclusive with seed bias
        max_iterations=5,
        min_similarity=0.15,            # Low threshold for refinement
        
        # String matching parameters
        substring_bonus=0.3,            # Bonus for partial matches
        seed_bias=0.1,                  # Bias toward seed clusters in refinement
        skip_phase2_if_few=True,        # Skip new cluster discovery if <50 unassigned
        force_complete=True             # Ensure 100% coverage
    )

In [7]:
import os
import re
import pandas as pd
from pathlib import Path

def retrieve_script(loc):
    local_path = loc.split("#")[0]
    content = open(local_path, "r", encoding="utf-8").read()
    return content

def preprocess_path(path):
    path_parts = path.split(os.sep)
    org_name, repo_name = path_parts[2].split("__")[0], path_parts[2].split("__")[1]
    file_path = os.path.join(*path_parts[3:]).replace(os.sep, "/").split(':')[0]
    file_path_url = f"https://github.com/{org_name}/{repo_name}/blob/{repo_commit_info[repo_name]}/{file_path}"
    return file_path_url

def python_grep(usage_pattern, repo_path, include_pattern=r"\.(py|sh|bash)", exclude_dirs=[]):
    """
    Python replacement for `grep -rn -E pattern directory --include=... --exclude-dir=...`
    
    Args:
        pattern (str): Regular expression pattern to search for
        directory (Path): Directory to search in
        include_pattern (str): File pattern to include
        exclude_dirs (list): List of directory names to exclude
    
    Returns:
        list: List of strings in format "filepath:line_number:line_content"
    """
    results = []
    
    # Compile the regex pattern
    usage_regex = re.compile(usage_pattern, re.MULTILINE)
    
    # Convert include pattern to regex
    include_regex = re.compile(include_pattern)
    
    # Walk through repo path recursively
    for root, dirs, files in os.walk(repo_path):
        # Remove excluded directories from dirs list to prevent walking into them
        dirs[:] = [d for d in dirs if d not in exclude_dirs]
        
        for file in files:
            # Check if file matches include pattern (use search instead of match)
            if not include_regex.search(file):
                continue
                
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                # Read all lines and iterate with line numbers
                for line_num, line in enumerate(f, 1):
                    if usage_regex.search(line):
                        # Format similar to grep output: filepath:line_number:line_content
                        result_line = f"{file_path}#L{line_num}:{line.strip()}"
                        results.append(result_line)
    
    return results

# Only exclude directories that contain Python files we DON'T want to search
exclude_dirs = [
    '.git',           # May contain .py files in hooks/scripts
    '__pycache__',    # Contains .pyc files (not .py, but just in case)
    'venv', '.venv',  # Virtual env contains installed packages (.py files)
    'site-packages', # Installed packages
    'node_modules'    # May contain .py files in some JS projects
]

df_harness = pd.read_csv("data/evaluation_harness_metadata.csv", encoding="utf-8")
df_harness['repo'] = df_harness['repo'].apply(lambda x: x.split(",")[0].split("/")[4] if ',' in x else x.split("/")[4])
repo_harness_mapping = dict(zip(df_harness["repo"], df_harness["harness"]))
repo_harness_mapping['evalai-cli'] = 'EvalAI'

df = []
path_repo = Path("data/repos")

for dependency, usage_pattern in usage_pattern_mapping.items():
    for repo_path in path_repo.iterdir():
        if ".json" in str(repo_path):
            continue

        dependent = repo_path.name.split("__")[-1]
        if repo_harness_mapping[dependent] == repo_harness_mapping[dependency]:
            continue
        
        # Use Python grep replacement instead of subprocess
        grep_results = python_grep(
            usage_pattern=usage_pattern,
            repo_path=repo_path,
            exclude_dirs=exclude_dirs
        )
        
        for line in grep_results:
            loc = preprocess_path(line)
            usecase_dict = {
                "dependent": repo_harness_mapping[dependent],
                "dependency": repo_harness_mapping[dependency],
                "line of code": loc,
                # "content": retrieve_script(line),
            }
            df.append(usecase_dict)

df = pd.DataFrame(df)
df.to_json("data/harness_dependencies.jsonl", orient="records", lines=True)

In [16]:
import os
import re
import pandas as pd
from pathlib import Path

def retrieve_script(loc):
    local_path = loc.split("#")[0]
    content = open(local_path, "r", encoding="utf-8").read()
    return content

def preprocess_path(path):
    path_parts = path.split(os.sep)
    org_name, repo_name = path_parts[2].split("__")[0], path_parts[2].split("__")[1]
    file_path = os.path.join(*path_parts[3:]).replace(os.sep, "/").split(':')[0]
    file_path_url = f"https://github.com/{org_name}/{repo_name}/blob/{repo_commit_info[repo_name]}/{file_path}"
    return file_path_url

def python_grep(usage_pattern, repo_path, include_pattern=r"\.(py|sh|bash)", exclude_dirs=[]):
    """
    Python replacement for `grep -rn -E pattern directory --include=... --exclude-dir=...`
    
    Args:
        pattern (str): Regular expression pattern to search for
        directory (Path): Directory to search in
        include_pattern (str): File pattern to include
        exclude_dirs (list): List of directory names to exclude
    
    Returns:
        list: List of strings in format "filepath:line_number:line_content"
    """
    results = []
    
    # Compile the regex pattern
    usage_regex = re.compile(usage_pattern, re.MULTILINE)
    
    # Convert include pattern to regex
    include_regex = re.compile(include_pattern)
    
    # Walk through repo path recursively
    for root, dirs, files in os.walk(repo_path):
        # Remove excluded directories from dirs list to prevent walking into them
        dirs[:] = [d for d in dirs if d not in exclude_dirs]
        
        for file in files:
            # Check if file matches include pattern (use search instead of match)
            if not include_regex.search(file):
                continue
                
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                # Read all lines and iterate with line numbers
                for line_num, line in enumerate(f, 1):
                    if usage_regex.search(line):
                        # Format similar to grep output: filepath:line_number:line_content
                        result_line = f"{file_path}#L{line_num}:{line.strip()}"
                        results.append(result_line)
    
    return results

# Only exclude directories that contain Python files we DON'T want to search
exclude_dirs = [
    '.git',           # May contain .py files in hooks/scripts
    '__pycache__',    # Contains .pyc files (not .py, but just in case)
    'venv', '.venv',  # Virtual env contains installed packages (.py files)
    'site-packages', # Installed packages
    'node_modules'    # May contain .py files in some JS projects
]

df_harness = pd.read_csv("data/evaluation_harness_metadata.csv", encoding="utf-8")
df_harness['repo'] = df_harness['repo'].apply(lambda x: x.split(","))
df_harness = df_harness.explode('repo')
df_harness['repo'] = df_harness['repo'].apply(lambda x: x.split("/")[4])
repo_harness_mapping = dict(zip(df_harness["repo"], df_harness["harness"]))
special_name_pattern_mapping = {
    'AlpacaEval': 'AlpacaEval2?',
}

df = []
path_repo = Path("data/repos")

for dependency, name_pattern in repo_harness_mapping.items():
    for repo_path in path_repo.iterdir():
        if ".json" in str(repo_path):
            continue

        dependent = repo_path.name.split("__")[-1]
        if repo_harness_mapping[dependent] == name_pattern:
            continue
        
        # Use Python grep replacement instead of subprocess
        grep_results = python_grep(
            usage_pattern=rf"\b{name_pattern}\b" if name_pattern not in special_name_pattern_mapping else rf"\b{special_name_pattern_mapping[name_pattern]}\b",
            repo_path=repo_path,
            exclude_dirs=exclude_dirs
        )
        
        for line in grep_results:
            loc = preprocess_path(line)
            usecase_dict = {
                "dependent": repo_harness_mapping[dependent],
                "dependency": name_pattern,
                "line of code": loc,
                # "content": retrieve_script(line),
            }
            df.append(usecase_dict)

df = pd.DataFrame(df)
df.to_json("data/harness_implicit_dependencies.jsonl", orient="records", lines=True)